# Part 2 — compare linear and cosine learning-rate decay

Start with **A0: warmup + linear decay** and **A1: warmup + cosine decay**. Both use the same small, two-token transformer, with each linear classifier supervised by the ground-truth label. This first comparison has **no teacher or knowledge distillation**. At evaluation, average the heads' softmax probabilities and report their negative log-likelihood (NLL), accuracy, and macro F1.

This is a new controlled comparison. The published Part 1 result used a different stopping/scheduling policy and a single MLP classifier; it is not the A0 result. The current Part 1 code supplies the six-block, width-192, four-head encoder configuration. These 100-epoch Imagenette runs are a local adaptation, not a reproduction of the 300-epoch ImageNet DeiT experiments.

Run the setup cells from the repository root, then execute A0 and A1 separately when ready. Each training cell starts a full 100-epoch run. No dataset download or full training has been executed in this notebook.

In [ ]:
import json
import random
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset

from datasets.imagenette import download_imagenette_dataset
from models.vision_transformer import build_distilled_vision_transformer
from pipeline.evaluate import evaluate_model
from pipeline.train import train_model

assert Path("models").is_dir(), "Run this notebook from the repository root."
SEED = 0
SPLIT_SEED = 0
EPOCHS = 100
WARMUP_EPOCHS = 5
BATCH_SIZE = 32
PEAK_LR = 5e-4
WEIGHT_DECAY = 0.01
DATA_ROOT = Path("data")
RESULTS_DIR = Path("artifacts") / datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_UTC") / "part2_schedule"
MODEL_CONFIG = dict(image_size=224, patch_size=16, hidden_dim=192, num_heads=4,
                    num_layers=6, mlp_ratio=4, num_classes=10, dropout=0.1)
device = torch.device("cuda" if torch.cuda.is_available() else
                      "mps" if torch.backends.mps.is_available() else "cpu")

def reset_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def save_json(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, default=lambda value: value.tolist()) + "\n")

print(f"Device: {device}; results: {RESULTS_DIR}")

## Keep the data fixed

Use only Imagenette's official **training** split, with deterministic shorter-side resize, center crop to 224, and pixel values in [0,1]. Reserve a stratified 20% of this pool for internal validation. The official validation split remains outside this notebook.

Save both the split indices and the dataset's ordered image list. The split seed and memberships remain fixed across A0 and A1. These shared deterministic transforms are intentional; augmentation belongs to a later experiment.

In [ ]:
pool = download_imagenette_dataset(root_dir=str(DATA_ROOT), split="train", image_size=224)
labels = np.array([label for _, label in pool._samples])
train_indices, validation_indices = train_test_split(
    np.arange(len(pool)), test_size=0.2, stratify=labels, random_state=SPLIT_SEED,
)
train_data = Subset(pool, train_indices.tolist())
validation_data = Subset(pool, validation_indices.tolist())
validation_loader = DataLoader(validation_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
np.savez(RESULTS_DIR / "split_indices.npz", train=train_indices, validation=validation_indices)
save_json(RESULTS_DIR / "dataset_manifest.json", {
    "dataset": "Imagenette", "distribution": "320px", "official_split": "train",
    "split_seed": SPLIT_SEED, "class_to_idx": pool.class_to_idx,
    "ordered_samples": [
        {"image": str(Path(path).relative_to(Path(pool._image_root))), "label": int(label)}
        for path, label in pool._samples
    ],
})
print(f"Training: {len(train_data)}; internal validation: {len(validation_data)}")

## Shared training procedure

Each call below starts a fresh model with the same initialization seed and a fresh training-loader generator. Resetting all random generators also matches dropout and data order across the two conditions as closely as the backend permits. Seeds do not guarantee bitwise reproducibility on every device.

Both conditions receive exactly 100 epochs, the same batch handling (including the final partial batch), five epochs of warmup, AdamW, peak LR 0.0005, weight decay 0.01, and dropout 0.1. The only changed setting is the decay shape. Both schedulers explicitly start warmup at `1e-6` and decay toward `1e-5`. The peak `5e-4` is the actual optimizer rate with no batch-size scaling; these values and the 100-epoch budget are local choices, not the complete published DeiT recipe.

The pipeline minimizes the equally weighted cross-entropy losses of the two heads. Keep its existing **highest validation accuracy** checkpoint rule for both runs. Full training continues after the best epoch because early stopping is disabled. Validation NLL uses the combined probability prediction; the training loss is a different objective, so do not interpret their difference as a pure generalization gap.

In [ ]:
histories, summaries = {}, {}

def run_schedule_experiment(run_id, schedule_name):
    reset_seed(SEED)
    train_loader = DataLoader(
        train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
        generator=torch.Generator().manual_seed(SEED), drop_last=False,
    )
    warmup_steps = WARMUP_EPOCHS * len(train_loader)
    model = build_distilled_vision_transformer(**MODEL_CONFIG)
    run_dir = RESULTS_DIR / run_id
    checkpoint = run_dir / "best.pth"
    config = dict(run_id=run_id, model=MODEL_CONFIG, student_seed=SEED, split_seed=SPLIT_SEED,
                  epochs=EPOCHS, batch_size=BATCH_SIZE, learning_rate=PEAK_LR,
                  weight_decay=WEIGHT_DECAY, scheduler=schedule_name,
                  warmup_steps=warmup_steps, warmup_lr=1e-6, min_lr=1e-5,
                  early_stopping_patience=None, supervision="both heads: ground-truth CE, equal weights",
                  evaluation="mean softmax probabilities", checkpoint_rule="highest validation accuracy",
                  device=str(device), torch_version=torch.__version__,
                  split_indices="../split_indices.npz", dataset_manifest="../dataset_manifest.json")
    save_json(run_dir / "config.json", config)
    history = train_model(
        model, train_loader, validation_loader, num_epochs=EPOCHS, learning_rate=PEAK_LR,
        weight_decay=WEIGHT_DECAY, device=device, checkpoint_path=checkpoint,
        scheduler=schedule_name, warmup_steps=warmup_steps, warmup_lr=1e-6, min_lr=1e-5,
        early_stopping_patience=None,
    )
    metrics = evaluate_model(model, validation_loader, device, checkpoint_path=checkpoint)
    summary = dict(run_id=run_id, parameters=sum(p.numel() for p in model.parameters()),
                   epochs_run=len(history["train_loss"]), best_epoch=history["best_epoch"],
                   validation=metrics, training_seconds=sum(history["train_seconds"]),
                   checkpoint=str(checkpoint))
    save_json(run_dir / "history.json", history)
    save_json(run_dir / "summary.json", summary)
    histories[run_id], summaries[run_id] = history, summary
    model.cpu()
    print(f"{run_id}: validation accuracy={metrics['accuracy']:.4f}, "
          f"NLL={metrics['loss']:.4f}, macro F1={metrics['macro_f1_score']:.4f}")
    return history, summary

## A0 — linear decay

Run this first. It establishes the new two-head, teacher-free reference; it does not reuse trained Part 1 weights.

In [ ]:
a0_history, a0_summary = run_schedule_experiment("A0_linear", "linear")

## A1 — cosine decay

Use the same configuration and split. This call rebuilds the model and resets the training-loader generator; A1 does not continue A0's trained checkpoint.

In [ ]:
a1_history, a1_summary = run_schedule_experiment("A1_cosine", "cosine_decay")

## Inspect the completed runs

Compare accuracy and NLL at the selected weights, the selected epochs, training time, and learning-rate curves. Differences from one seed remain exploratory. Checkpoints, configurations, histories, split membership, and summaries are saved under the displayed results directory. No test score is produced.

In [ ]:
if histories:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for run_id, history in histories.items():
        epochs = np.arange(1, len(history["train_loss"]) + 1)
        axes[0].plot(epochs, history["learning_rate"], label=run_id)
        axes[1].plot(epochs, history["val_loss"], label=run_id)
        axes[2].plot(epochs, history["val_accuracy"], label=run_id)
    for axis, title in zip(axes, ["LR at epoch start", "Validation NLL", "Validation accuracy"]):
        axis.set(title=title, xlabel="Epoch")
        axis.grid(True)
        axis.legend()
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / "schedule_comparison.png", dpi=160)
    plt.show()
    for run_id, summary in summaries.items():
        print(run_id, "selected epoch:", summary["best_epoch"], "validation:", summary["validation"])
else:
    print("No full training runs have been executed in this kernel.")

## Next experiments — planned, not implemented here

| Run | Next question |
| --- | --- |
| A2 | Does a stronger augmentation/regularization recipe improve the selected teacher-free setup? |
| C0 | Does two-token soft distillation help? |
| C1 | Does two-token hard distillation help under the same fixed protocol? |

Before implementing C0/C1, fix the teacher checkpoint, class mapping, loss weights, temperature where applicable, head receiving each target, and inference rule. Keep the teacher frozen and use only training images for gradient updates. These future conditions are not automatically the exact experiments reported in DeiT Table 3.

[DeiT paper](https://proceedings.mlr.press/v139/touvron21a.html) · [Published training settings, supplementary Table 9](https://proceedings.mlr.press/v139/touvron21a/touvron21a-supp.pdf)